# NB01 — Hospital Characteristics from CMS Data

**Purpose:** Build a hospital-level characteristics dataset for ~3,400 IPPS-participating acute care hospitals. This gives us the foundation for peer-group comparisons in later notebooks.

**What we're building:**
- A clean dataset with one row per hospital, containing: bed count, teaching status, ownership type (for-profit / nonprofit / government), urban vs. rural, state, CMS region
- Bed-size buckets (6 tiers) for peer-group matching in NB07

**Data Sources (two files, both free public downloads):**

1. **CMS Hospital General Information** — from the [Provider Data Catalog](https://data.cms.gov/provider-data/dataset/xubh-q36u). Contains hospital name, address, type, ownership, and quality ratings for every Medicare-certified hospital. Direct CSV download.

2. **CMS Provider of Services (POS) file** — from [data.cms.gov](https://data.cms.gov/provider-characteristics/hospitals-and-other-facilities). Contains detailed characteristics including bed count, teaching status, and urban/rural classification. Updated quarterly.

**Why two sources?** The Hospital General Information file has clean ownership labels and hospital type classifications. The POS file has bed counts and teaching indicators. We merge them on the CCN (CMS Certification Number) to get a complete picture. If the POS file is unavailable, we can supplement bed count and teaching status from the IPPS Impact File (NB02).

**Output:** `data/outputs/nb01_hospital_characteristics/hospital_characteristics.csv`

## 1. Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import requests
import os
import zipfile
import io
from pathlib import Path

PROJECT_ROOT = Path('..').resolve().parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'outputs' / 'nb01_hospital_characteristics'

RAW_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Raw data dir: {RAW_DIR}')
print(f'Output dir:   {OUTPUT_DIR}')

Project root: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap
Raw data dir: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap/data/raw
Output dir:   /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap/data/outputs/nb01_hospital_characteristics


## 2. Download Source 1: CMS Hospital General Information

### What is this dataset?

The **Hospital General Information** file is from CMS's Provider Data Catalog (the same system that powers Medicare's Care Compare website). It contains one row per Medicare-certified hospital with:

- **Facility ID** — The 6-digit CCN (CMS Certification Number), our key identifier across all CMS datasets
- **Facility Name** and full address (city, state, ZIP, county)
- **Hospital Type** — Acute Care, Critical Access, Children's, Psychiatric, etc.
- **Hospital Ownership** — Clean text labels like "Voluntary non-profit - Private", "Proprietary", "Government - State"
- **Emergency Services** — Whether the hospital has an ED
- **Hospital Overall Rating** — CMS Star Rating (1-5)

The direct CSV download link is:
`https://data.cms.gov/provider-data/api/1/datastore/query/xubh-q36u/0/download?format=csv`

If that URL stops working, search: **"Hospital General Information" on data.cms.gov** → download CSV

In [2]:
HGI_URL = 'https://data.cms.gov/provider-data/api/1/datastore/query/xubh-q36u/0/download?format=csv'
hgi_raw_path = RAW_DIR / 'hospital_general_info.csv'

if hgi_raw_path.exists():
    print(f'Already downloaded: {hgi_raw_path}')
    print(f'File size: {hgi_raw_path.stat().st_size / 1e6:.1f} MB')
else:
    print('Downloading CMS Hospital General Information...')
    try:
        resp = requests.get(HGI_URL, timeout=120)
        resp.raise_for_status()
        if len(resp.content) > 10000 and (b',' in resp.content[:1000]):
            with open(hgi_raw_path, 'wb') as f:
                f.write(resp.content)
            print(f'Downloaded {len(resp.content) / 1e6:.1f} MB')
        else:
            raise ValueError('Response does not look like CSV')
    except Exception as e:
        print(f'Download failed: {e}')
        print('\n=== MANUAL DOWNLOAD ===')
        print('1. Go to: https://data.cms.gov/provider-data/dataset/xubh-q36u')
        print('2. Click the Download CSV button')
        print(f'3. Save as: {hgi_raw_path}')
        print('\nOR search: "Hospital General Information" on data.cms.gov')

Already downloaded: /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap/data/raw/hospital_general_info.csv
File size: 1.6 MB


In [3]:
df_hgi = pd.read_csv(hgi_raw_path, dtype=str, low_memory=False)
print(f'Hospital General Info: {len(df_hgi):,} rows, {len(df_hgi.columns)} columns')
print(f'\nColumns:')
for i, col in enumerate(df_hgi.columns):
    sample = df_hgi[col].dropna().iloc[0] if df_hgi[col].notna().any() else 'N/A'
    print(f'  {i:2d}. {col:45s} sample: {str(sample)[:40]}')

Hospital General Info: 5,426 rows, 38 columns

Columns:
   0. Facility ID                                   sample: 010001
   1. Facility Name                                 sample: SOUTHEAST HEALTH MEDICAL CENTER
   2. Address                                       sample: 1108 ROSS CLARK CIRCLE
   3. City/Town                                     sample: DOTHAN
   4. State                                         sample: AL
   5. ZIP Code                                      sample: 36301
   6. County/Parish                                 sample: HOUSTON
   7. Telephone Number                              sample: (334) 793-8701
   8. Hospital Type                                 sample: Acute Care Hospitals
   9. Hospital Ownership                            sample: Government - Hospital District or Author
  10. Emergency Services                            sample: Yes
  11. Meets criteria for birthing friendly designation sample: Y
  12. Hospital overall rating                       

## 3. Download Source 2: CMS Provider of Services (POS) File — QIES

The **Provider of Services (POS)** file is CMS's master registry of every Medicare-certified facility. As of 2025, it's been renamed to **"Provider of Services File: QIES: Hospital & Non-Hospital Data"** (QIES = Quality Improvement and Evaluation System).

Key columns we need:
- **`PRVDR_NUM`** — Provider number (CCN), our merge key
- **`BED_CNT`** / **`CRTFD_BED_CNT`** — Total and certified bed counts
- **`RSDNT_PGM_ALPTHC_SW`** / **`RSDNT_PGM_OSTPTHC_SW`** — Residency programs (Y/N = teaching indicator)
- **`MDCL_SCHL_AFLTN_CD`** — Medical school affiliation code (1-4 scale)
- **`CBSA_URBN_RRL_IND`** — Urban/Rural classification (U/R)
- **`GNRL_CNTL_TYPE_CD`** — General control type (ownership code)

**Download approaches (in order):**
1. **CSV download:** `https://data.cms.gov/data-api/v1/dataset/8ba0f9b4-9493-4aa0-9f82-44ea9468d1b5/data.csv`
2. **JSON API (paginated):** `https://data.cms.gov/data-api/v1/dataset/8ba0f9b4-9493-4aa0-9f82-44ea9468d1b5/data?size=500&offset=0`
3. **Manual:** Go to [data.cms.gov → Provider Characteristics → Hospitals → Provider of Services File - QIES](https://data.cms.gov/provider-characteristics/hospitals-and-other-facilities/provider-of-services-file-quality-improvement-and-evaluation-system) → click Download

**If you can't get the POS file at all:** That's OK — we'll derive bed count and teaching status from the IPPS Impact File in NB02.

In [4]:
# POS QIES dataset UUID (verified from data.cms.gov API modal)
POS_DATASET_UUID = '8ba0f9b4-9493-4aa0-9f82-44ea9468d1b5'

# Primary: CSV bulk download
POS_CSV_URL = f'https://data.cms.gov/data-api/v1/dataset/{POS_DATASET_UUID}/data.csv'

# Fallback: JSON API (paginated — we'll fetch in chunks of 5000)
POS_JSON_URL = f'https://data.cms.gov/data-api/v1/dataset/{POS_DATASET_UUID}/data'

# Legacy URLs (may still work for older quarterly snapshots)
POS_LEGACY_URLS = [
    'https://data.cms.gov/provider-data/api/1/datastore/query/4pq5-n9py/0/download?format=csv',
    'https://data.cms.gov/sites/default/files/2025-01/POS_OTHER_Jan25.csv',
    'https://data.cms.gov/sites/default/files/2024-10/POS_OTHER_Oct24.csv',
]

# Check for any POS / QIES file already in RAW_DIR (handles manual downloads)
pos_raw_path = None
pos_downloaded = False

# Look for manually downloaded QIES file first, then our standard name
for candidate in sorted(RAW_DIR.glob('POS_File_QIES*.csv'), reverse=True):
    if candidate.stat().st_size > 1_000_000:
        pos_raw_path = candidate
        pos_downloaded = True
        print(f'Found manually downloaded POS file: {candidate.name} ({candidate.stat().st_size / 1e6:.1f} MB)')
        break

if pos_raw_path is None:
    pos_raw_path = RAW_DIR / 'pos_hospital.csv'
    if pos_raw_path.exists() and pos_raw_path.stat().st_size > 1_000_000:
        print(f'POS file exists: {pos_raw_path} ({pos_raw_path.stat().st_size / 1e6:.1f} MB)')
        pos_downloaded = True

if not pos_downloaded:
    # --- Attempt 1: CSV bulk download ---
    print('Attempt 1: CSV bulk download from QIES endpoint...')
    pos_raw_path = RAW_DIR / 'pos_hospital.csv'
    try:
        resp = requests.get(POS_CSV_URL, timeout=300, stream=True)
        resp.raise_for_status()
        size = int(resp.headers.get('Content-Length', 0))
        print(f'  Response: HTTP {resp.status_code}, Content-Length: {size:,} bytes')
        if size > 1_000_000 or resp.headers.get('Content-Type','').startswith('text/csv'):
            with open(pos_raw_path, 'wb') as f:
                for chunk in resp.iter_content(65536):
                    f.write(chunk)
            actual_size = pos_raw_path.stat().st_size
            if actual_size > 1_000_000:
                print(f'  Downloaded {actual_size / 1e6:.1f} MB')
                pos_downloaded = True
            else:
                print(f'  File too small ({actual_size} bytes) — removing')
                pos_raw_path.unlink()
    except Exception as e:
        print(f'  Failed: {e}')
    
    # --- Attempt 2: JSON API with pagination ---
    if not pos_downloaded:
        print('\nAttempt 2: JSON API with pagination...')
        try:
            test = requests.get(f'{POS_JSON_URL}?size=1', timeout=30)
            test.raise_for_status()
            print(f'  API responsive. Fetching all records in chunks of 5000...')
            
            all_records = []
            offset = 0
            page_size = 5000
            while True:
                url = f'{POS_JSON_URL}?size={page_size}&offset={offset}'
                r = requests.get(url, timeout=120)
                r.raise_for_status()
                batch = r.json()
                if not batch:
                    break
                all_records.extend(batch)
                print(f'  Fetched {len(all_records):,} records...', end='\r')
                if len(batch) < page_size:
                    break
                offset += page_size
            
            print(f'\n  Total: {len(all_records):,} records')
            if len(all_records) > 1000:
                df_pos_json = pd.DataFrame(all_records)
                df_pos_json.to_csv(pos_raw_path, index=False)
                print(f'  Saved {pos_raw_path.stat().st_size / 1e6:.1f} MB')
                pos_downloaded = True
        except Exception as e:
            print(f'  JSON API failed: {e}')
    
    # --- Attempt 3: Legacy URLs ---
    if not pos_downloaded:
        print('\nAttempt 3: Legacy POS_OTHER URLs...')
        for url in POS_LEGACY_URLS:
            short = url.split('/')[-1][:50]
            print(f'  Trying: {short}...')
            try:
                resp = requests.get(url, timeout=300, stream=True)
                if resp.status_code == 200:
                    size = int(resp.headers.get('Content-Length', 0))
                    if size > 1_000_000:
                        with open(pos_raw_path, 'wb') as f:
                            for chunk in resp.iter_content(65536):
                                f.write(chunk)
                        print(f'  Downloaded {pos_raw_path.stat().st_size / 1e6:.1f} MB')
                        pos_downloaded = True
                        break
                    else:
                        print(f'  Too small ({size} bytes)')
                else:
                    print(f'  HTTP {resp.status_code}')
            except Exception as e:
                print(f'  {type(e).__name__}')
    
    if not pos_downloaded:
        print('\n=== MANUAL DOWNLOAD ===')
        print('Go to: https://data.cms.gov/provider-characteristics/hospitals-and-other-facilities/')
        print('       provider-of-services-file-quality-improvement-and-evaluation-system')
        print('Click "Download" → "Latest Dataset Only" → "Download Files"')
        print(f'Save CSV to: {RAW_DIR}/')
        print('\nOr: bed count + teaching will come from NB02 Impact File.')

Found manually downloaded POS file: POS_File_QIES_Q4_2025.csv (115.8 MB)


In [5]:
df_pos = None
if pos_downloaded and pos_raw_path.exists():
    df_pos = pd.read_csv(pos_raw_path, dtype=str, low_memory=False)
    print(f'POS file: {len(df_pos):,} rows, {len(df_pos.columns)} columns')
    print(f'Source: {pos_raw_path.name}')
    print('Columns:')
    for i, col in enumerate(df_pos.columns):
        print(f'  {i:3d}. {col}')
else:
    print('POS not available - will use NB02 Impact File for beds/teaching.')

POS file: 77,522 rows, 473 columns
Source: POS_File_QIES_Q4_2025.csv
Columns:
    0. PRVDR_CTGRY_SBTYP_CD
    1. PRVDR_CTGRY_CD
    2. CHOW_CNT
    3. CHOW_DT
    4. CITY_NAME
    5. ACPTBL_POC_SW
    6. CMPLNC_STUS_CD
    7. SSA_CNTY_CD
    8. CROSS_REF_PROVIDER_NUMBER
    9. CRTFCTN_DT
   10. ELGBLTY_SW
   11. FAC_NAME
   12. INTRMDRY_CARR_CD
   13. MDCD_VNDR_NUM
   14. ORGNL_PRTCPTN_DT
   15. CHOW_PRIOR_DT
   16. INTRMDRY_CARR_PRIOR_CD
   17. PRVDR_NUM
   18. RGN_CD
   19. SKLTN_REC_SW
   20. STATE_CD
   21. SSA_STATE_CD
   22. STATE_RGN_CD
   23. ST_ADR
   24. PHNE_NUM
   25. PGM_TRMNTN_CD
   26. TRMNTN_EXPRTN_DT
   27. CRTFCTN_ACTN_TYPE_CD
   28. GNRL_CNTL_TYPE_CD
   29. ZIP_CD
   30. FIPS_STATE_CD
   31. FIPS_CNTY_CD
   32. CBSA_URBN_RRL_IND
   33. CBSA_CD
   34. ACRDTN_EFCTV_DT
   35. ACRDTN_EXPRTN_DT
   36. ACRDTN_TYPE_CD
   37. TOT_AFLTD_AMBLNC_SRVC_CNT
   38. TOT_AFLTD_ASC_CNT
   39. TOT_COLCTD_HOSP_CNT
   40. TOT_AFLTD_ESRD_CNT
   41. TOT_AFLTD_FQHC_CNT
   42. TOT_AFLTD_HHA_

## 4. Filter to IPPS Acute Care Hospitals

The Hospital General Info file contains ALL Medicare hospitals. We filter to **short-term acute care** — the ~3,400 that participate in IPPS, get paid via MS-DRGs, and have a Case Mix Index.

We exclude: Critical Access (CAH), psychiatric, children's, long-term care (LTCH), and VA hospitals.

In [6]:
type_col = None
for col in df_hgi.columns:
    if 'TYPE' in col.upper() and 'HOSPITAL' in col.upper():
        type_col = col
        break
if type_col is None:
    type_cols = [c for c in df_hgi.columns if 'TYPE' in c.upper()]
    if type_cols:
        type_col = type_cols[0]

if type_col:
    print(f'Type column: "{type_col}"')
    print(df_hgi[type_col].value_counts().to_string())
    acute_mask = df_hgi[type_col].str.upper().str.contains('ACUTE', na=False)
    df_acute = df_hgi[acute_mask].copy()
    print(f'\nFiltered: {len(df_acute):,} acute care (from {len(df_hgi):,} total)')
else:
    print('No type column found. Keeping all.')
    df_acute = df_hgi.copy()

Type column: "Hospital Type"
Hospital Type
Acute Care Hospitals                    3116
Critical Access Hospitals               1376
Psychiatric                              633
Acute Care - Veterans Administration     132
Childrens                                 94
Rural Emergency Hospital                  39
Acute Care - Department of Defense        32
Long-term                                  4

Filtered: 3,280 acute care (from 5,426 total)


## 5. Extract & Standardize Key Fields

In [7]:
def find_col(df, keywords, exclude=None):
    exclude = exclude or []
    for col in df.columns:
        col_upper = col.upper().replace(' ', '_')
        if any(kw in col_upper for kw in keywords):
            if not any(ex in col_upper for ex in exclude):
                return col
    return None

hgi_col_map = {
    'ccn': find_col(df_acute, ['FACILITY_ID', 'PROVIDER_ID', 'CCN', 'PRVDR_NUM']),
    'hospital_name': find_col(df_acute, ['FACILITY_NAME', 'HOSPITAL_NAME', 'PROVIDER_NAME']),
    'city': find_col(df_acute, ['CITY', 'TOWN']),
    'state': find_col(df_acute, ['STATE'], exclude=['FIPS', 'COUNTY']),
    'zip_code': find_col(df_acute, ['ZIP', 'ZIPCODE']),
    'county': find_col(df_acute, ['COUNTY', 'PARISH']),
    'ownership': find_col(df_acute, ['OWNERSHIP', 'CONTROL']),
    'emergency_services': find_col(df_acute, ['EMERGENCY']),
    'overall_rating': find_col(df_acute, ['OVERALL_RATING', 'STAR', 'RATING']),
}

print('Column mapping:')
for field, col in hgi_col_map.items():
    print(f'  {field:25s} {f"-> {col}" if col else "NOT FOUND"}')

rename_map = {v: k for k, v in hgi_col_map.items() if v is not None}
df_hosp = df_acute[list(rename_map.keys())].rename(columns=rename_map).copy()
df_hosp['ccn'] = df_hosp['ccn'].astype(str).str.strip().str.zfill(6)

print(f'\n{len(df_hosp):,} acute care hospitals')
df_hosp.head()

Column mapping:
  ccn                       -> Facility ID
  hospital_name             -> Facility Name
  city                      -> City/Town
  state                     -> State
  zip_code                  -> ZIP Code
  county                    -> County/Parish
  ownership                 -> Hospital Ownership
  emergency_services        -> Emergency Services
  overall_rating            -> Hospital overall rating

3,280 acute care hospitals


,ccn,hospital_name,city,state,zip_code,county,ownership,emergency_services,overall_rating
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,DOTHAN,AL,36301,HOUSTON,Government - Hospital District or Authority,Yes,4
1,010005,MARSHALL MEDICAL CENTERS,BOAZ,AL,35957,MARSHALL,Government - Hospital District or Authority,Yes,3
2,010006,NORTH ALABAMA MEDICAL CENTER,FLORENCE,AL,35630,LAUDERDALE,Proprietary,Yes,2
3,010007,MIZELL MEMORIAL HOSPITAL,OPP,AL,36467,COVINGTON,Voluntary non-profit - Private,Yes,1
4,010008,CRENSHAW COMMUNITY HOSPITAL,LUVERNE,AL,36049,CRENSHAW,Proprietary,Yes,Not Available


## 6. Standardize Ownership

Three categories: **Nonprofit** (60%), **For-Profit** (~25%), **Government** (~15%). For-profit hospitals invest more in CDI since documentation accuracy directly impacts shareholder returns.

In [8]:
if 'ownership' in df_hosp.columns:
    print('Raw ownership values:')
    print(df_hosp['ownership'].value_counts().to_string())
    
    def categorize_ownership(val):
        if pd.isna(val): return 'Other'
        val = str(val).upper()
        if any(kw in val for kw in ['PROPRIETARY', 'FOR-PROFIT', 'FOR PROFIT', 'INVESTOR']): return 'For-Profit'
        elif any(kw in val for kw in ['VOLUNTARY', 'NON-PROFIT', 'NONPROFIT', 'CHURCH', 'PRIVATE']): return 'Nonprofit'
        elif any(kw in val for kw in ['GOVERNMENT', 'STATE', 'FEDERAL', 'LOCAL', 'DISTRICT', 'TRIBAL']): return 'Government'
        else: return 'Other'
    
    df_hosp['ownership_category'] = df_hosp['ownership'].apply(categorize_ownership)
    print('\nCategories:')
    for cat, count in df_hosp['ownership_category'].value_counts().items():
        print(f'  {cat:15s}: {count:5,} ({count/len(df_hosp)*100:.1f}%)')
else:
    df_hosp['ownership_category'] = 'Unknown'

Raw ownership values:
ownership
Voluntary non-profit - Private                 1485
Proprietary                                     651
Voluntary non-profit - Other                    253
Government - Hospital District or Authority     228
Voluntary non-profit - Church                   206
Government - Local                              140
Veterans Health Administration                  132
Physician                                        70
Government - State                               51
Department of Defense                            32
Government - Federal                             25
Tribal                                            7

Categories:
  Nonprofit      : 1,944 (59.3%)
  For-Profit     :   651 (19.8%)
  Government     :   451 (13.8%)
  Other          :   234 (7.1%)


## 7. Add Bed Count & Teaching Status

Hospital General Info doesn't have beds or teaching. We get these from:
- **Option A:** POS file (if downloaded)
- **Option B:** IPPS Impact File from NB02 (IME > 0 = teaching hospital)

In [9]:
beds_added = False
teaching_added = False

# Option A: POS file
if df_pos is not None:
    print('Using POS file for beds & teaching...')
    
    # --- Explicit QIES column names (preferred) then fuzzy fallback ---
    # CCN: PRVDR_NUM is the primary provider number in QIES
    pos_ccn = None
    for candidate in ['PRVDR_NUM', 'PROVIDER_NUM', 'PROV_NO']:
        if candidate in df_pos.columns:
            pos_ccn = candidate
            break
    if pos_ccn is None:
        pos_ccn = find_col(df_pos, ['PRVDR_NUM', 'PROVIDER_NUM', 'CCN', 'PROVIDER_ID'],
                           exclude=['CROSS_REF', 'PARENT', 'RELATED', 'MEDICARE_MEDICAID',
                                    'MEDICARE_HOSPICE', 'FQHC'])
    
    # Beds: BED_CNT (total beds) or CRTFD_BED_CNT (certified beds)
    pos_bed = None
    for candidate in ['BED_CNT', 'CRTFD_BED_CNT']:
        if candidate in df_pos.columns:
            pos_bed = candidate
            break
    if pos_bed is None:
        pos_bed = find_col(df_pos, ['BED_CNT', 'BEDS', 'NUMBER_OF_BEDS'],
                           exclude=['OVRRD', 'AIDS', 'ALZHMR', 'DLYS', 'DSBL', 'HEAD',
                                    'HOSPC', 'HNTGTN', 'REHAB', 'VNTLTR', 'PSYCH',
                                    'ICFIID', 'MDCD_NF', 'MDCR_SNF', 'MDCR_MDCD'])
    
    # Urban/Rural: CBSA_URBN_RRL_IND
    pos_urban = None
    for candidate in ['CBSA_URBN_RRL_IND']:
        if candidate in df_pos.columns:
            pos_urban = candidate
            break
    if pos_urban is None:
        pos_urban = find_col(df_pos, ['URBAN', 'RURAL', 'URBN_RRL', 'CBSA_URBN'])
    
    print(f'  CCN column:     {pos_ccn}')
    print(f'  Beds column:    {pos_bed}')
    print(f'  Urban/Rural:    {pos_urban}')
    
    if pos_ccn:
        lk = df_pos[[pos_ccn]].copy()
        lk['ccn'] = lk[pos_ccn].astype(str).str.strip().str.zfill(6)
        
        if pos_bed:
            lk['beds'] = pd.to_numeric(df_pos[pos_bed], errors='coerce')
            # Sanity check
            valid_beds = lk['beds'].dropna()
            print(f'  Beds stats: n={len(valid_beds):,}, mean={valid_beds.mean():.0f}, '
                  f'median={valid_beds.median():.0f}, max={valid_beds.max():.0f}')
        
        if pos_urban:
            lk['urban_raw'] = df_pos[pos_urban]
        
        # Teaching: combine residency program flags
        teach_cols = [c for c in ['RSDNT_PGM_ALPTHC_SW', 'RSDNT_PGM_OSTPTHC_SW',
                                   'RSDNT_PGM_DNTL_SW', 'RSDNT_PGM_PDTRC_SW',
                                   'RSDNT_PGM_OTHR_SW'] if c in df_pos.columns]
        
        if teach_cols:
            lk['is_teaching_pos'] = False
            for tc in teach_cols:
                lk['is_teaching_pos'] = lk['is_teaching_pos'] | (df_pos[tc].str.upper() == 'Y')
            print(f'  Teaching from residency flags: {lk["is_teaching_pos"].sum():,} providers')
        elif 'MDCL_SCHL_AFLTN_CD' in df_pos.columns:
            afltn = pd.to_numeric(df_pos['MDCL_SCHL_AFLTN_CD'], errors='coerce')
            lk['is_teaching_pos'] = afltn.isin([1, 2, 3])
            print(f'  Teaching from affiliation code: {lk["is_teaching_pos"].sum():,} providers')
        
        cols = ['ccn'] + [c for c in ['beds', 'is_teaching_pos', 'urban_raw'] if c in lk.columns]
        lk = lk[cols].drop_duplicates(subset='ccn')
        df_hosp = df_hosp.merge(lk, on='ccn', how='left')
        
        if 'beds' in df_hosp.columns and df_hosp['beds'].notna().any():
            beds_added = True
            print(f'  Merged beds: {df_hosp["beds"].notna().sum():,} / {len(df_hosp):,} hospitals')
        else:
            print(f'  WARNING: No bed data after merge (0 matches)')
        
        if 'is_teaching_pos' in df_hosp.columns:
            teaching_added = True
            print(f'  Merged teaching: {df_hosp["is_teaching_pos"].sum():,} teaching hospitals')
else:
    print('POS not available - trying NB02 Impact File...')

# Option B: Impact File fallback
if not beds_added or not teaching_added:
    for p in [RAW_DIR / 'ipps_impact_file.xlsx', RAW_DIR / 'ipps_impact_file.csv']:
        if p.exists():
            print(f'Loading Impact File: {p.name}')
            try:
                if p.suffix == '.xlsx':
                    xl = pd.ExcelFile(p)
                    for s in xl.sheet_names:
                        t = pd.read_excel(p, sheet_name=s, dtype=str)
                        if len(t) > 100:
                            df_imp = t
                            break
                else:
                    df_imp = pd.read_csv(p, dtype=str, low_memory=False)
                ic = find_col(df_imp, ['PROV_NO', 'PROVIDER_NO', 'PRVDR_NUM', 'CCN', 'PROVIDER_ID'])
                ib = find_col(df_imp, ['BED', 'BEDS'], exclude=['ADJ'])
                ii = find_col(df_imp, ['IME', 'TEACH'])
                if ic:
                    il = pd.DataFrame()
                    il['ccn'] = df_imp[ic].astype(str).str.strip().str.zfill(6)
                    if ib and not beds_added:
                        il['beds'] = pd.to_numeric(df_imp[ib], errors='coerce')
                    if ii and not teaching_added:
                        il['ime_pct'] = pd.to_numeric(df_imp[ii], errors='coerce')
                    il = il.drop_duplicates(subset='ccn')
                    if 'beds' in il.columns and not beds_added:
                        df_hosp = df_hosp.merge(il[['ccn', 'beds']], on='ccn', how='left')
                        beds_added = True
                        print(f'  Beds added: {df_hosp["beds"].notna().sum():,}')
                    if 'ime_pct' in il.columns and not teaching_added:
                        df_hosp = df_hosp.merge(il[['ccn', 'ime_pct']], on='ccn', how='left')
                        teaching_added = True
                        print(f'  IME added: {df_hosp["ime_pct"].notna().sum():,}')
            except Exception as e:
                print(f'  Error: {e}')
            break
    else:
        if not beds_added:
            print('No Impact File yet. Run NB02 first, then re-run this cell.')

Using POS file for beds & teaching...
  CCN column:     PRVDR_NUM
  Beds column:    BED_CNT
  Urban/Rural:    CBSA_URBN_RRL_IND
  Beds stats: n=25,727, mean=76, median=20, max=5709
  Teaching from residency flags: 2,696 providers
  Merged beds: 3,274 / 3,280 hospitals
  Merged teaching: 1,167 teaching hospitals


In [10]:
# Finalize beds and teaching
if not beds_added:
    df_hosp['beds'] = np.nan
    print('Beds: not available yet (run NB02, then re-run cell above)')

if not teaching_added:
    if 'hospital_name' in df_hosp.columns:
        nm = df_hosp['hospital_name'].fillna('').str.upper()
        df_hosp['is_teaching'] = nm.str.contains(
            'UNIVERSITY|MEDICAL CENTER|TEACHING|ACADEMIC|MAYO|JOHNS HOPKINS', regex=True)
        print(f'Teaching estimated from names: {df_hosp["is_teaching"].sum():,} (approximate)')
    else:
        df_hosp['is_teaching'] = False
else:
    if 'is_teaching_pos' in df_hosp.columns:
        # Direct boolean from POS residency program flags
        df_hosp['is_teaching'] = df_hosp['is_teaching_pos'].fillna(False)
        df_hosp.drop(columns=['is_teaching_pos'], inplace=True)
    elif 'teach_raw' in df_hosp.columns:
        tv = df_hosp['teach_raw'].astype(str).str.upper()
        df_hosp['is_teaching'] = tv.isin(['Y','YES','1','TRUE','T']) | (pd.to_numeric(df_hosp['teach_raw'], errors='coerce').fillna(0) > 0)
        df_hosp.drop(columns=['teach_raw'], inplace=True)
    elif 'ime_pct' in df_hosp.columns:
        df_hosp['is_teaching'] = df_hosp['ime_pct'].fillna(0).astype(float) > 0
        df_hosp.drop(columns=['ime_pct'], inplace=True)

print(f'Teaching: {df_hosp["is_teaching"].sum():,} | Non-teaching: {(~df_hosp["is_teaching"]).sum():,}')

Teaching: 1,167 | Non-teaching: 2,113


/var/folders/0d/xqmptt6x035b25m88r2ffc9m0000gn/T/ipykernel_93900/2404644985.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_hosp['is_teaching'] = df_hosp['is_teaching_pos'].fillna(False)


## 8. Bed-Size Tiers, Region, Peer Groups

In [11]:
# Bed-size tiers
bins = [0, 24, 99, 199, 399, 599, float('inf')]
labels = ['1-24', '25-99', '100-199', '200-399', '400-599', '600+']
if df_hosp['beds'].notna().sum() > 0:
    df_hosp['bed_size_tier'] = pd.cut(df_hosp['beds'], bins=bins, labels=labels, include_lowest=True)
    print('Bed-size distribution:')
    for t, c in df_hosp['bed_size_tier'].value_counts().sort_index().items():
        print(f'  {str(t):10s}: {c:5,} ({c/len(df_hosp)*100:.1f}%)')
else:
    df_hosp['bed_size_tier'] = np.nan
    print('Bed tiers pending (need bed data from NB02).')

# Census regions
REGIONS = {
    'Northeast': ['CT','ME','MA','NH','RI','VT','NJ','NY','PA'],
    'Midwest': ['IL','IN','MI','OH','WI','IA','KS','MN','MO','NE','ND','SD'],
    'South': ['DE','FL','GA','MD','NC','SC','VA','DC','WV','AL','KY','MS','TN','AR','LA','OK','TX'],
    'West': ['AZ','CO','ID','MT','NV','NM','UT','WY','AK','CA','HI','OR','WA']
}
st_to_r = {s: r for r, ss in REGIONS.items() for s in ss}
df_hosp['state'] = df_hosp['state'].str.strip().str.upper()
df_hosp['census_region'] = df_hosp['state'].map(st_to_r)
print(f'\nRegions:')
for r, c in df_hosp['census_region'].value_counts().items():
    print(f'  {r:12s}: {c:5,}')

# Urban/Rural
if 'urban_raw' in df_hosp.columns:
    df_hosp['is_urban'] = df_hosp['urban_raw'].astype(str).str.upper().str.strip().isin(['U','URBAN','1','Y'])
    df_hosp.drop(columns=['urban_raw'], inplace=True)
else:
    df_hosp['is_urban'] = True
print(f'\nUrban: {df_hosp["is_urban"].sum():,} | Rural: {(~df_hosp["is_urban"]).sum():,}')

# Peer groups
if df_hosp['beds'].notna().sum() > 0:
    df_hosp['peer_group'] = (
        df_hosp['bed_size_tier'].astype(str) + '_' +
        df_hosp['is_teaching'].map({True:'teaching', False:'non_teaching'}) + '_' +
        df_hosp['ownership_category'].str.lower().str.replace('-','_').str.replace(' ','_')
    )
    print(f'\nPeer groups: {df_hosp["peer_group"].nunique()}')
else:
    df_hosp['peer_group'] = 'pending_bed_data'
    print('\nPeer groups pending bed data.')

Bed-size distribution:
  1-24      :   152 (4.6%)
  25-99     :   767 (23.4%)
  100-199   :   790 (24.1%)
  200-399   :   887 (27.0%)
  400-599   :   377 (11.5%)
  600+      :   301 (9.2%)



Regions:
  South       : 1,370
  Midwest     :   736
  West        :   640
  Northeast   :   472

Urban: 2,568 | Rural: 712



Peer groups: 49


## 9. Summary & Save

In [12]:
print('=' * 60)
print('HOSPITAL CHARACTERISTICS SUMMARY')
print('=' * 60)
print(f'Hospitals:  {len(df_hosp):,}')
print(f'States:     {df_hosp["state"].nunique()}')
print(f'Ownership:  {dict(df_hosp["ownership_category"].value_counts())}')
print(f'Teaching:   {df_hosp["is_teaching"].sum():,} / {len(df_hosp):,}')
if df_hosp['beds'].notna().any():
    print(f'Beds:       mean={df_hosp["beds"].mean():.0f}, median={df_hosp["beds"].median():.0f}')
print(f'\nMissing:')
m = df_hosp.isnull().sum()
print(m[m > 0].to_string() if m.any() else '  None')
print(f'\nTop 10 states:')
for s, c in df_hosp['state'].value_counts().head(10).items():
    print(f'  {s}: {c}')

HOSPITAL CHARACTERISTICS SUMMARY
Hospitals:  3,280
States:     56
Ownership:  {'Nonprofit': 1944, 'For-Profit': 651, 'Government': 451, 'Other': 234}
Teaching:   1,167 / 3,280
Beds:       mean=268, median=185

Missing:
beds              6
bed_size_tier     6
census_region    62

Top 10 states:
  TX: 292
  CA: 290
  FL: 178
  PA: 140
  NY: 139
  OH: 123
  IL: 119
  GA: 100
  MI: 94
  NC: 86


In [13]:
out_cols = ['ccn','hospital_name','city','state','zip_code','county',
            'beds','bed_size_tier','ownership','ownership_category',
            'is_teaching','is_urban','census_region','peer_group','overall_rating']
final = [c for c in out_cols if c in df_hosp.columns]
df_out = df_hosp[final].copy()

out_path = OUTPUT_DIR / 'hospital_characteristics.csv'
df_out.to_csv(out_path, index=False)

print(f'Saved {len(df_out):,} hospitals -> {out_path}')
print(f'Size: {out_path.stat().st_size / 1e6:.2f} MB')
print(f'Columns: {list(df_out.columns)}')
print(f'\n NB01 complete. Joins with NB02 + NB03 via CCN.')
if df_out['beds'].isna().all():
    print('  Note: Run NB02 for bed count, then re-run cells 7-8 here.')

Saved 3,280 hospitals -> /Users/trinidadcisneros/Documents/Development/Coding/bitterscientist.com/bitterscientist.com/folders/ds_blogs/projects/smarterdx_documentation_gap/data/outputs/nb01_hospital_characteristics/hospital_characteristics.csv
Size: 0.54 MB
Columns: ['ccn', 'hospital_name', 'city', 'state', 'zip_code', 'county', 'beds', 'bed_size_tier', 'ownership', 'ownership_category', 'is_teaching', 'is_urban', 'census_region', 'peer_group', 'overall_rating']

 NB01 complete. Joins with NB02 + NB03 via CCN.
